In [0]:
%sql
use catalog `p&l_silver`

In [0]:

# Level 0 - BASE RAW SUMS (Building Blocks)

from pyspark.sql.functions import *
# Using alternate_group

#Total Food
total_food = abs(sum(when(col("alternate_group").like("Total Food"), col("amount")).otherwise(0.0)))

#Total Beverage
total_beverage = abs(sum(when(col("alternate_group").like("Total Beverage"), col("amount")).otherwise(0.0)))

total_beverage_wine = abs(sum(when(col("alternate_group").like("Total Beverage - Wine"), col("amount")).otherwise(0.0)))

total_beverage_spirits = abs(sum(when(col("alternate_group").like("Total Beverage - Spirits"), col("amount")).otherwise(0.0)))

total_beverage_tobacco = abs(sum(when(col("alternate_group").like("Total Beverage - tobacco"), col("amount")).otherwise(0.0)))

total_retail = abs(sum(when(col("alternate_group").like("Total Retail"), col("amount")).otherwise(0.0)))

total_other = abs(sum(when(col("alternate_group").like("Total Other"), col("amount")).otherwise(0.0)))

# Using management_group
total_net_restaurant_sales = abs(sum(when(col("management_group").like("Total Net Restaurant sales"), col("amount")).otherwise(0.0)))

total_cost_of_sales = abs(sum(when(col("management_group").like("TOTAL COST OF SALES"), col("amount")).otherwise(0.0)))

total_wages = abs(sum(when(col("management_group").like("Total  Wages"), col("amount")).otherwise(0.0)))

total_salary_related = abs(sum(when(col("management_group").like("Total Salary Related"), col("amount")).otherwise(0.0)))

total_marketing = abs(sum(when(col("management_group").like("Total Marketing"), col("amount")).otherwise(0.0)))

total_pr_entertainment = abs(sum(when(col("management_group").like("Total PR & Entertainment"), col("amount")).otherwise(0.0)))

total_operational_rm = abs(sum(when(col("management_group").like("Total Opertional R&M"), col("amount")).otherwise(0.0))) + \
                       abs(sum(when(col("management_group").like("Total Operational R&M"), col("amount")).otherwise(0.0)))

total_rm_it = abs(sum(when(col("management_group").like("Total R&M IT"), col("amount")).otherwise(0.0)))

total_overheads = abs(sum(when(col("management_group").like("Total Overheads"), col("amount")).otherwise(0.0)))

total_commissions = abs(sum(when(col("management_group").like("Total Commissions"), col("amount")).otherwise(0.0)))

total_travelling_expenses = abs(sum(when(col("management_group").like("Total Travelling Expenses"), col("amount")).otherwise(0.0)))

total_administration = abs(sum(when(col("management_group").like("Total Administration"), col("amount")).otherwise(0.0)))

total_utilities = abs(sum(when(col("management_group").like("Total Utilities"), col("amount")).otherwise(0.0)))

total_non_controllables_rent = abs(sum(when(col("management_group").like("TOTAL NON-CONTROLLABLES RENT"), col("amount")).otherwise(0.0)))

total_non_controllables_oths = abs(sum(when(col("management_group").like("TOTAL -  NON-CONTROLLABLES - OTHS"), col("amount")).otherwise(0.0)))

total_other_income = abs(sum(when(col("management_group").like("Total Other Income"), col("amount")).otherwise(0.0)))

total_rental_income = abs(sum(when(col("management_group").like("Total Rental Income"), col("amount")).otherwise(0.0)))

total_insurance_claim = abs(sum(when(col("management_group").like("Total -Insurance Claim"), col("amount")).otherwise(0.0)))

total_interest_income = abs(sum(when(col("management_group").like("Total Interest- Income"), col("amount")).otherwise(0.0)))

total_tax = abs(sum(when(col("management_group").like("Total Tax"), col("amount")).otherwise(0.0)))

total_interest = abs(sum(when(col("management_group").like("Total Interest"), col("amount")).otherwise(0.0)))

total_impairment = abs(sum(when(col("management_group").like("Total Impairment"), col("amount")).otherwise(0.0)))

total_depreciation = abs(sum(when(col("management_group").like("Total Depreciation"), col("amount")).otherwise(0.0)))

total_amortization = abs(sum(when(col("management_group").like("Total Amortization"), col("amount")).otherwise(0.0)))

total_ifrs = abs(sum(when(col("management_group").like("Total IFRS"), col("amount")).otherwise(0.0)))

# Level 1 using Level 0 blocks
gross_margin = total_net_restaurant_sales - total_cost_of_sales

total_personal_cost = total_salary_related + total_wages

total_r_m = total_rm_it + total_operational_rm

grand_total_marketing = total_pr_entertainment + total_marketing

grand_total_other_income = total_other_income + total_interest_income + total_insurance_claim + total_rental_income

total_dep_amort = total_amortization + total_depreciation

# Level 2 using level 1 blocks

gross_profit = gross_margin - total_personal_cost

total_controllable_costs = (
    total_utilities +
    total_administration +
    total_overheads +
    total_r_m +
    grand_total_marketing +
    total_commissions +
    total_travelling_expenses
)

total_non_controllables = total_non_controllables_rent + total_non_controllables_oths

# Level 3 using level 2 blocks

controllable_profit = gross_profit - total_controllable_costs

store_operating_profit = controllable_profit - total_non_controllables

proratable_expenses = (
    total_personal_cost +           # Total Personnel Cost
    grand_total_marketing +         # Total Marketing (includes PR & Entertainment)
    total_r_m +                     # Total R&M
    total_travelling_expenses +
    abs(sum(when(col("mapped_name").like("%Printing & Stationery%"), col("amount")).otherwise(0.0))) +
    abs(sum(when(col("mapped_name").like("%Telephone Expenses%"), col("amount")).otherwise(0.0))) +
    abs(sum(when(col("mapped_name").like("%Other Admin Expenses%"), col("amount")).otherwise(0.0))) +
    abs(sum(when(col("mapped_name").like("%Licenses & Permits%"), col("amount")).otherwise(0.0))) +
    abs(sum(when(col("mapped_name").like("%Insurance%"), col("amount")).otherwise(0.0))) +
    total_non_controllables_rent
)

# Final level
pre_opening_exp = when(
    abs(total_net_restaurant_sales) == 0, 
    -store_operating_profit
).otherwise(
    when(
        # Check if this is the store opening month
        (col("month") == upper(date_format(
            try_to_date(col("store_open_date2"), "yyyy-mm-dd"), 
            "MMM"
        ))) & 
        (year(col("year")) == year(try_to_date(col("store_open_date2"), "yyyy-mm-dd"))),
        
        # === NON-OPERATING DAYS ===
        (dayofmonth(try_to_date(col("store_open_date2"), "yyyy-mm-dd")) - 1) * (
            (total_personal_cost +
             total_overheads +
             grand_total_marketing +
             total_r_m +
             total_travelling_expenses +
             abs(sum(when(col("mapped_name").like("Printing & Stationery"), col("amount")).otherwise(0.0))) +
             abs(sum(when(col("mapped_name").like("Telephone expenses"), col("amount")).otherwise(0.0))) +
             abs(sum(when(col("mapped_name").like("Other Admin Exp."), col("amount")).otherwise(0.0))) +
             abs(sum(when(col("mapped_name").like("License and permits"), col("amount")).otherwise(0.0))) +
             abs(sum(when(col("mapped_name").like("Insurances"), col("amount")).otherwise(0.0))) +
             total_non_controllables_rent
            ) / dayofmonth(last_day(concat(col("year"), lit("-"), col("month"), lit("-01"))))
        )
    ).otherwise(lit(0.0))
)

rev_pre_opening_exp = abs(sum(when(col("mapped_name").like("%Rev. Pre-Opening Expenses%"), col("amount")).otherwise(0.0)))

brand_ho_allocation = abs(sum(when(col("mapped_name").like("%Brand HO allocation%"), col("amount")).otherwise(0.0)))

# 4-Wall EBITDA
four_wall_ebitda = store_operating_profit + grand_total_other_income + pre_opening_exp + brand_ho_allocation


# STORE EBITDA
store_ebitda = store_operating_profit + rev_pre_opening_exp + pre_opening_exp + grand_total_other_income


# 4-Wall Net Profit
four_wall_net_profit = four_wall_ebitda - total_depreciation

# STORE NET PROFIT / LOSS
store_net_profit_loss = store_ebitda - total_tax - total_interest - total_impairment - total_dep_amort - total_ifrs

# Store EBITDA after Pre-Opening Expenses
store_ebitda_after_pre_opening_expenses =  store_ebitda - pre_opening_exp

# Store Net Profit after Pre-Opening Expenses
store_net_profit_after_pre_opening_expsense = store_ebitda_after_pre_opening_expenses -total_tax - total_interest - total_impairment - total_dep_amort - total_ifrs


In [0]:

# {path}/src/foodquest_pnl.py
from pyspark.sql.functions import *
import re, time
from pyspark.sql.window import Window

def to_snake_case(name):
    return re.sub(r'[\s\-]+', '_', name).lower()

def to_snake_case_df(df):
    for col_name in df.columns:
        df = df.withColumnRenamed(col_name, to_snake_case(col_name))
    return df

def create_total_row(df, group_cols, total_col_name, total_label, dimension_cols):
    """
    Create aggregated total rows (subgroup, group, major group totals).
    
    Parameters:
    - df: Input dataframe
    - group_cols: List of columns to group by (e.g., ['location', 'month', 'year', 'sub_group'])
    - total_col_name: Column name to use for the total label (e.g., 'sub_group')
    - total_label: Label type ('Total' or 'Grand Total')
    - dimension_cols: List of dimension columns to preserve
    """
    return df.groupBy(*group_cols).agg(
        sum("amount").alias("amount")
    ).select(
        col("location"),
        col("month"),
        col("year"),
        # lit(None).alias("account_number"),
        lit(None).alias("account_name"),
        # concat(lit("Total "),col(total_col_name)).alias("name"),
        concat(lit(""), col(total_col_name)).alias("mapped_name"),
        lit(None).cast("string").alias("account_type"),
        lit(None).alias("major_group"),
        lit(None).alias("group"),
        lit(None).alias("sub_group"),
        lit(None).alias("management_group"),
        col("amount"),
        *[col(c) for c in dimension_cols],
        lit(total_label).alias("Detail/Total")
    )


def create_calculated_metric(df, metric_name, calculation_expr, dimension_cols, total_label="Grand Total"):
    """
    Create calculated metric rows (Gross Profit, Operating Profit, EBITDA, Net Profit).
    
    Parameters:
    - df: Input dataframe
    - metric_name: Name of the metric (e.g., 'Gross Profit')
    - calculation_expr: PySpark column expression for the calculation
    - dimension_cols: List of dimension columns to preserve
    - total_label: Label type (default 'Grand Total')
    """
    group_cols = ["location", "month", "year"] + dimension_cols
    
    return df.groupBy(*group_cols).agg(
        calculation_expr.alias("amount")
    ).select(
        col("location"),
        col("month"),
        col("year"),
        lit(None).alias("account_name"),
        lit(metric_name).alias("mapped_name"),
        lit(None).cast("string").alias("account_type"),
        lit(None).alias("major_group"),
        lit(None).alias("group"),
        lit(None).alias("sub_group"),
        lit(None).alias("management_group"),
        col("amount"),
        *[col(c) for c in dimension_cols],
        lit(total_label).alias("Detail/Total")
    )


def add_previous_year_data(df):
    """Add previous year (PY) amounts via self-join."""
    df_current = df.withColumn("year", col("year"))
    
    df_py = df_current.alias("py").select(
        (col("year") + 1).alias("year_join"),
        col("netsuite_location_name").alias("py_location"),
        col("mapped_name").alias("py_mapped_name"),
        col("amount").alias("py_amount"),
        col("month").alias("py_month")
    )
    
    return df_current.alias("curr").join(
        df_py,
        (col("curr.year") == col("year_join")) &  
        (col("curr.netsuite_location_name") == col("py_location")) &  
        (col("curr.mapped_name") == col("py_mapped_name")) &  
        (col("curr.month") == col("py_month")),
        "left"
    ).select(
        col("curr.*"),
        coalesce(col("py_amount"), lit(0.0)).alias("py_amount")
    )


def add_net_sales_calculations(df):
    """Add actual and PY net sales calculations at location, brand, and company levels."""
    window_location = Window.partitionBy("location", "year", "month")
    window_brand = Window.partitionBy("brand_id", "year", "month")
    window_company = Window.partitionBy("company_id", "year", "month")
    
    sales_condition = col("account_name").like("Total Sales")
    
    return df \
        .withColumn("store_actual_net_sales",
            sum(when(sales_condition, col("amount")).otherwise(0.0)).over(window_location)) \
        .withColumn("store_py_net_sales",
            sum(when(sales_condition, col("py_amount")).otherwise(0.0)).over(window_location)) \
        .withColumn("brand_act_net_sales",
            sum(when(sales_condition, col("amount")).otherwise(0.0)).over(window_brand)) \
        .withColumn("brand_py_net_sales",
            sum(when(sales_condition, col("py_amount")).otherwise(0.0)).over(window_brand)) \
        .withColumn("company_act_net_sales",
            sum(when(sales_condition, col("amount")).otherwise(0.0)).over(window_company)) \
        .withColumn("company_py_net_sales",
            sum(when(sales_condition, col("py_amount")).otherwise(0.0)).over(window_company))


def get_dimension_columns():
    """Return list of standard dimension columns used throughout transformations."""
    return [
        "netsuite_location_name", "type", "location_id", "brand_id",
        "company_id", "parent_company", "country_code", "zone", "store_type", "city", "store_open_date2"
    ]

def join_dataframes(base_df, join_configs):
    """
    Perform multiple joins on a base dataframe.
    
    Parameters:
    -----------
    base_df : DataFrame
        The base dataframe to join other dataframes to
    join_configs : list of dict
        List of join configurations, each containing:
        - 'df': DataFrame to join
        - 'left_key': column name or expression from left/base df
        - 'right_key': column name or expression from right df
        - 'join_type': 'inner', 'left', 'right', 'outer', etc.
        
    Returns:
    --------
    DataFrame: Result of all joins applied sequentially
    
    Example:
    --------
    join_configs = [
        {
            'df': df_coa_master,
            'left_key': col("accountNumber").cast("string"),
            'right_key': df_coa_master["account_number"].cast("string"),
            'join_type': 'inner'
        },
        {
            'df': df_location_master,
            'left_key': col("location"),
            'right_key': df_location_master["netsuite_location_name"],
            'join_type': 'left'
        }
    ]
    result = join_dataframes(df, join_configs)
    """
    result_df = base_df
    
    for config in join_configs:
        result_df = result_df.join(
            config['df'],
            config['left_key'] == config['right_key'],
            config['join_type']
        )
    
    return result_df

def final_df(df_all_masters):
    dimension_cols = get_dimension_columns()
        
    # =============================================
    # LEVEL 1 KPIs
    # =============================================
    df_detail = df_all_masters.select(
        col("location"), col("month"), col("year"),
        col("account_name"), col("mapped_name"), col("account_type"),
        col("major_group"), col("group"), col("sub_group"), col("management_group"),
        col("amount"),
        *[col(c) for c in dimension_cols],
        lit("Detail").alias("Detail/Total")
    )

    df_alternate_group_totals = create_total_row(
            df_all_masters,
            ["location", "month", "year", "alternate_group"] + dimension_cols,
            "alternate_group",
            "Total",
            dimension_cols
        )

    df_management_group_totals = create_total_row(
            df_all_masters,
            ["location", "month", "year", "management_group"] + dimension_cols,
            "management_group",
            "Total",
            dimension_cols
        )

    df_gross_margin = create_calculated_metric(
        df_detail, "Gross Margin", gross_margin, dimension_cols
    )

    df_total_personal_cost = create_calculated_metric(
        df_detail, "Total Personal cost", total_personal_cost, dimension_cols
    )

    df_total_r_m = create_calculated_metric(
        df_detail, "Total R & M", total_r_m, dimension_cols
    )

    df_grand_total_marketing = create_calculated_metric(
        df_detail, "Grand Total Marketing", grand_total_marketing, dimension_cols
    )

    df_grand_total_other_income = create_calculated_metric(
        df_detail, "Grand Total other Income", grand_total_other_income, dimension_cols
    )
    # Total Depreiciation & Amortization
    df_total_dep_amort = create_calculated_metric(
        df_detail, "Total Depreciation & Amortization", total_dep_amort, dimension_cols
    )

    # =============================================
    # LEVEL 2 KPIs
    # =============================================

    df_gross_profit = create_calculated_metric(
        df_detail, "GROSS PROFIT", gross_profit, dimension_cols
    )

    df_total_controllable_costs = create_calculated_metric(
        df_detail, "TOTAL CONTROLLABLE COSTS", total_controllable_costs, dimension_cols
    )

    df_total_non_controllables = create_calculated_metric(
        df_detail, "TOTAL NON-CONTROLLABLES", total_non_controllables, dimension_cols
    )

    # =============================================
    # LEVEL 3 KPIs
    # =============================================

    df_controllable_profit = create_calculated_metric(
        df_detail, "CONTROLLABLE PROFIT", controllable_profit, dimension_cols
    )

    df_store_operating_profit = create_calculated_metric(
        df_detail, "STORE OPERATING PROFIT", store_operating_profit, dimension_cols
    )

    # =============================================
    # FINAL LEVEL KPIs
    # =============================================

    df_4wall_ebitda = create_calculated_metric(
        df_detail, "4-Wall EBITDA", four_wall_ebitda, dimension_cols
    )

    df_store_ebitda = create_calculated_metric(
        df_detail, "STORE EBITDA", store_ebitda, dimension_cols
    )

    df_4wall_net_profit = create_calculated_metric(
        df_detail, "4-Wall Net Profit", four_wall_net_profit, dimension_cols
    )

    df_store_net_profit = create_calculated_metric(
        df_detail, "STORE NET PROFIT/LOSS", store_net_profit_loss, dimension_cols
    )

    df_pre_opening_exp = create_calculated_metric(
        df_detail, "Pre Opening Expenses", 
        pre_opening_exp, dimension_cols
    )

    df_store_ebitda_after_pre = create_calculated_metric(
        df_detail, "Store EBITDA afer Preopening Expenses", 
        store_ebitda_after_pre_opening_expenses, dimension_cols
    )

    df_store_net_profit_after_pre = create_calculated_metric(
        df_detail, "Store Net Profit afer Preopening Expenses", 
        store_net_profit_after_pre_opening_expsense, dimension_cols
    )

    df_all_kpis = (df_detail
                .unionAll(df_alternate_group_totals)
                .unionAll(df_management_group_totals)
                .unionAll(df_gross_margin)
                .unionAll(df_total_personal_cost)
                .unionAll(df_total_r_m)
                .unionAll(df_grand_total_marketing)
                .unionAll(df_grand_total_other_income)
                .unionAll(df_total_dep_amort)
                .unionAll(df_gross_profit)
                .unionAll(df_total_controllable_costs)
                .unionAll(df_total_non_controllables)
                .unionAll(df_controllable_profit)
                .unionAll(df_store_operating_profit)
                .unionAll(df_4wall_ebitda)
                .unionAll(df_store_ebitda)
                .unionAll(df_4wall_net_profit)
                .unionAll(df_store_net_profit)
                .unionAll(df_pre_opening_exp)
                .unionAll(df_store_ebitda_after_pre)
                .unionAll(df_store_net_profit_after_pre))

    df_sort = spark.read.table('default.coa_master_management')
    df_sort = to_snake_case_df(df_sort)
    df_sort = df_sort.dropDuplicates(["mapped_name"])
    df_sort = df_sort.select("mapped_name", "management_sort_order", "management_details_total")
    df_final_sort = df_all_kpis.join(
        df_sort, 
        df_sort["mapped_name"].cast("string") == df_all_kpis.mapped_name,
        'inner'
    ).drop(df_sort["mapped_name"]).select('*'
    ).orderBy(
            "parent_company", "company_id", "brand_id", 
            "netsuite_location_name", 'year', 'month', 'management_sort_order'
        )
    
    return df_final_sort

print("All variables and functions from other-notebook are now available")


In [0]:
from pyspark.sql.functions import *

# Read both tables individually
df1 = spark.read.table("default.postman_response_january")
df2 = spark.read.table("default.postman_response_january_2025")

# Combine them into one DataFrame
df = df1.unionByName(df2)

# df = spark.read.table('default.postman_response_january')
df_exploded = df.select(explode('results').alias('result')).select('result.*')

# ✅ Reverse sign when accountType == 'Income'
df_exploded = df_exploded.withColumn(
    "amount",
    when(col("accountType") == "Income", -col("amount"))
    .when(col("accountName") == "75704 Talabat- Commission Discount", -col("amount"))
    .when(col("accountName") == "75713 Noon- Commission Discount", -col("amount"))
    .otherwise(col("amount"))
)

# Load brand allocation data
df_brand_allocation_cost = spark.read.table('default.brand_ho_allocation_cost_updated')

# Inspect schemas to understand column types
print("=== df_exploded schema ===")
df_exploded.printSchema()
print(f"\nColumn count: {len(df_exploded.columns)}")

print("\n=== df_brand_allocation_cost schema ===")
df_brand_allocation_cost.printSchema()

# Based on the schema, transform brand allocation to match exploded format
# Use when() to handle "-" values and convert to NULL
df_brand_ho_rows = df_brand_allocation_cost.select(
    lit("73219").cast("string").alias("accountNo"),
    lit("73219 - Brand HO allocation cost").cast("string").alias("accountName"),
    col("netsuite_location_name").cast("string").alias("location"),
    # ✅ Use when() to convert "-" to NULL, then cast to double
    when(col("Brand_Ho_Allocation_Cost") == "-", lit(None))
        .otherwise(col("Brand_Ho_Allocation_Cost"))
        .cast("double")
        .alias("amount"),
    col("month").cast("string").alias("month"),
    col("year").cast("string").alias("year")
).withColumn("accounttype", lit("Expense").cast("string"))

# Filter out rows where amount is NULL (from malformed "-" values)
df_brand_ho_rows = df_brand_ho_rows.filter(col("amount").isNotNull())

# Add any other columns from df_exploded as nulls
target_columns = df_exploded.columns
for column in target_columns:
    if column not in df_brand_ho_rows.columns:
        # Get the data type from df_exploded
        col_type = [f.dataType for f in df_exploded.schema if f.name == column][0]
        df_brand_ho_rows = df_brand_ho_rows.withColumn(column, lit(None).cast(col_type))

# Reorder columns to match df_exploded
df_brand_ho_rows = df_brand_ho_rows.select(df_exploded.columns)

print("\n=== df_brand_ho_rows final schema ===")
df_brand_ho_rows.printSchema()

# Now union should work
df_exploded = df_exploded.union(df_brand_ho_rows)

print(f"\n✅ Union successful! Total rows: {df_exploded.count()}")
df_exploded.display()

In [0]:
# df = spark.read.table('default.postman_response_january')

# df_exploded = df.select(explode('results').alias('result')).select('result.*')

df_coa_master = spark.read.table("default.coa_master_management")
df_coa_master = to_snake_case_df(df_coa_master)
df_location_master = spark.read.table("default.dim_location_master")
df_location_master = to_snake_case_df(df_location_master)
# df_coa_master_distinct = df_coa_master.dropDuplicates(["mapped_name"]) # Distinct new_grouping

from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType

for column in df_coa_master.columns:
    if dict(df_coa_master.dtypes)[column] == 'string':
        df_coa_master = df_coa_master.withColumn(column, trim(col(column)))

df_all_masters = df_exploded.join(
        df_coa_master, 
        df_coa_master["account_number"].cast("string") == df_exploded["accountNo"], 
        'inner'
    ).join(
        df_location_master,
        col("location") == df_location_master.netsuite_location_name,
        'left'
    )

df_final_actual = final_df(df_all_masters)
df_final_actual = df_final_actual.withColumn('year', year(col('year')))


In [0]:
df_final_actual.filter(col('month') == 'JAN').filter(col('location').isNotNull()).display()